# XGBoost: who wins given that a seat changes?

Train an XGBoost classifier **only on eligible seats that changed party**, then evaluate destination accuracy on actual changed seats in **2005, 2010, 2015, 2017 and 2019**. Each evaluation election is held out from both fitting and hyperparameter selection. Only `TEST_TRAIN/train.csv` is read; 2024 is not used.

To make this comparable with the conditional model in `06_exploring_multilayer_model.ipynb`, reuse its eligibility rules and pre-election role features. Predict the winning challenger role (`contesting_party`, `third_party`, `fourth_party`, or `oth`) and map it back to the constituency's party. This guarantees that the predicted destination cannot be the incumbent. This notebook measures the destination **assuming a change occurs**; it does not estimate whether a seat will change.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "TEST_TRAIN" / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")
raw = pd.read_csv(ROOT / "TEST_TRAIN" / "train.csv")
EVALUATION_YEARS = [2005, 2010, 2015, 2017, 2019]
PARTIES = ["con", "lib", "lab", "natSW"]
ROLES = ["incumbent", "contesting_party", "third_party", "fourth_party"]
display(raw.groupby("election").size().rename("available_rows").to_frame())

,available_rows
election,
1987,633
1992,634
1997,641
2001,641
2005,628
2010,632
2015,632
2017,632
2019,632


## Eligibility and role-based features

Previous winners labelled `oth` cannot be mapped because there is no corresponding previous share column. Exclude those rows, missing previous winners, missing outcome labels, and incomplete previous party shares (needed to rank challengers). **Keep current `oth` winners** when the previous winner is supported: these are valid seat changes. The audit below reports coverage by election.

`natSW` is the dataset's combined SNP/Plaid category. There are no polling or projected-share fields for this category. Missing numeric role-specific values are left as missing for XGBoost to handle natively. Existing projections are used as provided, without clipping or recomputing them. The upstream SQL uses previous national *seat* shares in its projection formula, so these should not be interpreted as calibrated vote forecasts.

Current-election shares, majority and winner are never predictors. The previous majority and region are retained. Party identities and the national governing party are categorical predictors.

In [2]:
previous_columns = [f"previous_{p}_share" for p in PARTIES]
known_previous = raw["previous_winner"].isin(PARTIES)
complete_previous = raw[previous_columns].notna().all(axis=1)
known_outcome = raw["winner"].notna()
eligible = known_previous & complete_previous & known_outcome
coverage = raw.assign(
    eligible=eligible,
    unsupported_previous=~known_previous,
    incomplete_previous=~complete_previous,
    missing_outcome=~known_outcome,
).groupby("election").agg(
    available=("eligible", "size"), included=("eligible", "sum"),
    unsupported_previous=("unsupported_previous", "sum"),
    incomplete_previous=("incomplete_previous", "sum"),
    missing_outcome=("missing_outcome", "sum"),
)
coverage["excluded"] = coverage["available"] - coverage["included"]
display(coverage)  # Exclusion reasons can overlap.
data = raw.loc[eligible].copy().reset_index(drop=True)


def engineer_features(frame):
    """Use only pre-election fields; preserve each party's role across families."""
    shares = frame[previous_columns].to_numpy(dtype=float)
    incumbent_index = pd.Index(PARTIES).get_indexer(frame["previous_winner"])
    if (incumbent_index < 0).any() or not np.isfinite(shares).all():
        raise ValueError("Role mapping needs supported previous winners and complete shares.")
    ranked = np.argsort(-shares, axis=1, kind="stable")
    challengers = ranked[ranked != incumbent_index[:, None]].reshape(-1, 3)
    role_indices = np.column_stack([incumbent_index, challengers])
    families = {
        "share": previous_columns,
        "projected_share": ["projected_con_share", "projected_lib_share", "projected_lab_share", None],
        "polling": ["Conservative", "LD", "Labour", None],
        "previous_national_share": [f"previous_nat_{p}_share" for p in PARTIES],
    }
    features = pd.DataFrame(index=frame.index)
    for role_number, role in enumerate(ROLES):
        features[f"{role}_party"] = np.asarray(PARTIES)[role_indices[:, role_number]]
        for suffix, columns in families.items():
            values = np.column_stack([
                frame[col].to_numpy(dtype=float) if col else np.full(len(frame), np.nan)
                for col in columns
            ])
            features[f"{role}_{suffix}"] = values[np.arange(len(frame)), role_indices[:, role_number]]
    features["previous_share_gap"] = features["incumbent_share"] - features["contesting_party_share"]
    features["projected_share_gap"] = features["incumbent_projected_share"] - features["contesting_party_projected_share"]
    features["previous_majority_proportion"] = frame["previous_majority_proportion"]
    features["country/region"] = frame["country/region"]
    features["national_governing_party"] = frame["incumbent"]
    return features

X = engineer_features(data)
y = data["winner"].ne(data["previous_winner"]).astype(int).rename("seat_changed")
assert X["incumbent_party"].equals(data["previous_winner"].rename("incumbent_party"))
assert X["incumbent_party"].ne(X["contesting_party_party"]).all()
# Changing the outcome cannot change the features.
pd.testing.assert_frame_equal(X, engineer_features(data.drop(columns=["winner"])))


,available,included,unsupported_previous,incomplete_previous,missing_outcome,excluded
election,,,,,,
1987,633,633,0,0,0,0
1992,634,633,1,1,0,1
1997,641,550,91,91,0,91
2001,641,639,2,2,0,2
2005,628,626,2,2,0,2
2010,632,631,1,1,0,1
2015,632,630,2,0,0,2
2017,632,628,4,1,0,4
2019,632,629,2,3,0,3


## Changed-seat sample and destination labels

Keep only rows where `winner != previous_winner`. Previous winning parties outside `con`, `lib`, `lab`, and `natSW` are excluded because their previous shares cannot be mapped. Current `oth` winners remain valid destinations. The audit below counts changed seats by election and destination role; these are observations, not distinct constituencies across time. No current result is an input feature.

In [3]:
DESTINATION_ROLES = ["contesting_party", "third_party", "fourth_party", "oth"]
changed = y.eq(1)
changed_data = data.loc[changed].reset_index(drop=True)
changed_X = X.loc[changed].reset_index(drop=True)
destination = pd.Series(pd.NA, index=changed_data.index, dtype="object", name="destination")
for role in DESTINATION_ROLES[:-1]:
    destination.loc[changed_data["winner"].eq(changed_X[f"{role}_party"])] = role
destination.loc[changed_data["winner"].eq("oth")] = "oth"
assert destination.notna().all()
assert changed_data["winner"].ne(changed_data["previous_winner"]).all()
display(pd.crosstab(changed_data["election"], destination).reindex(columns=DESTINATION_ROLES, fill_value=0))


destination,contesting_party,third_party,fourth_party,oth
election,,,,
1987,38,6,0,0
1992,48,2,1,0
1997,148,10,0,2
2001,20,0,0,2
2005,50,5,0,2
2010,107,2,0,2
2015,85,12,10,2
2017,58,7,0,0
2019,74,0,0,1


## Grid search using earlier elections only

Search all 16 combinations of `max_depth = [2, 3, 4, 5]` and `n_estimators = [25, 50, 100, 200]`, keeping `learning_rate=0.05`. This explores shallow trees and a range of boosting rounds for a relatively small changed-seat sample. Other controls are fixed, with no class weighting. This is a bounded search, not a claim of globally optimal settings.

For each held-out election, use expanding-window validation within the earlier data: train on elections before an inner validation year and validate on that entire year. Select the combination with the highest mean validation accuracy, giving each inner election equal weight, then refit on all earlier changed seats. Role accuracy equals destination-party accuracy because challenger roles map uniquely to parties within a seat. Preprocessing and target encoding are fitted afresh inside every fold. A destination absent from training cannot be predicted, but is retained when scoring validation and evaluation rows.

Numeric missing values are handled natively by XGBoost; categorical values are imputed and one-hot encoded inside the pipeline. Single-class inner folds use the only observed role as a constant prediction. Fixed seeds and one XGBoost thread keep the run reproducible and resource usage bounded.

References: [XGBoost estimator API](https://xgboost.readthedocs.io/en/stable/python/python_api.html) and [GridSearchCV custom splits](https://scikit-learn.org/1.7/modules/generated/sklearn.model_selection.GridSearchCV.html).


In [4]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier


class DestinationXGB(ClassifierMixin, BaseEstimator):
    """Encode only the classes observed in this training fold."""
    def __init__(self, max_depth=3, n_estimators=100):
        self.max_depth = max_depth
        self.n_estimators = n_estimators

    def fit(self, X, y):
        self.encoder_ = LabelEncoder().fit(y)
        self.classes_ = self.encoder_.classes_
        self.n_features_in_ = X.shape[1]
        self.model_ = None
        if len(self.classes_) > 1:
            objective_params = (
                {"objective": "binary:logistic", "eval_metric": "logloss"}
                if len(self.classes_) == 2 else
                {"objective": "multi:softprob", "num_class": len(self.classes_),
                 "eval_metric": "mlogloss"}
            )
            self.model_ = XGBClassifier(
                max_depth=self.max_depth, n_estimators=self.n_estimators,
                learning_rate=0.05, **objective_params,
                tree_method="hist", n_jobs=1, random_state=42,
            )
            self.model_.fit(X, self.encoder_.transform(y))
        return self

    def predict(self, X):
        check_is_fitted(self, "classes_")
        if self.model_ is None:
            return np.repeat(self.classes_[0], X.shape[0])
        return self.encoder_.inverse_transform(self.model_.predict(X).astype(int))


def make_destination_model():
    categorical = changed_X.select_dtypes(include=["object", "string"]).columns.tolist()
    numeric = changed_X.columns.difference(categorical).tolist()
    preprocessing = ColumnTransformer([
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numeric", "passthrough", numeric),
    ])
    return Pipeline([("preprocess", preprocessing), ("classifier", DestinationXGB())])


PARAM_GRID = {
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__n_estimators": [25, 50, 100, 200],
}
searches = {}
models = {}
metric_rows = []
prediction_frames = []
fold_rows = []
for year in EVALUATION_YEARS:
    training = changed_data["election"].lt(year)
    evaluation = changed_data["election"].eq(year)
    train_X = changed_X.loc[training].reset_index(drop=True)
    train_y = destination.loc[training].reset_index(drop=True)
    train_years = changed_data.loc[training, "election"].reset_index(drop=True)
    splits = []
    for validation_year in sorted(train_years.unique())[1:]:
        train_idx = np.flatnonzero(train_years.lt(validation_year))
        validation_idx = np.flatnonzero(train_years.eq(validation_year))
        assert train_years.iloc[train_idx].max() < validation_year < year
        splits.append((train_idx, validation_idx))
        fold_rows.append({"evaluation_election": year, "validation_election": validation_year,
                          "n_train": len(train_idx), "n_validation": len(validation_idx)})
    if not splits or not evaluation.any():
        raise ValueError(f"Insufficient training or evaluation data for {year}")
    search = GridSearchCV(make_destination_model(), PARAM_GRID, scoring="accuracy",
                          cv=splits, refit=True, n_jobs=1, error_score="raise")
    search.fit(train_X, train_y)
    searches[year] = search
    models[year] = search.best_estimator_
    predicted_roles = search.predict(changed_X.loc[evaluation])
    heldout = changed_data.loc[evaluation].copy()
    predicted_parties = np.full(len(heldout), "oth", dtype=object)
    for role in DESTINATION_ROLES[:-1]:
        matches = predicted_roles == role
        predicted_parties[matches] = changed_X.loc[evaluation, f"{role}_party"].to_numpy()[matches]
    assert (predicted_parties != heldout["previous_winner"].to_numpy()).all()
    accuracy = accuracy_score(heldout["winner"], predicted_parties)
    assert np.isclose(accuracy, accuracy_score(destination.loc[evaluation], predicted_roles))
    metric_rows.append({
        "election": year, "n_changed_train": int(training.sum()),
        "n_changed_evaluation": int(evaluation.sum()), "n_inner_folds": len(splits),
        "best_max_depth": search.best_params_["classifier__max_depth"],
        "best_n_estimators": search.best_params_["classifier__n_estimators"],
        "mean_cv_accuracy": search.best_score_,
        "n_correct": int((predicted_parties == heldout["winner"].to_numpy()).sum()),
        "changed_seat_accuracy": accuracy,
        "always_contesting_accuracy": accuracy_score(
            heldout["winner"], changed_X.loc[evaluation, "contesting_party_party"]),
    })
    prediction_frames.append(heldout[["constituency_name", "election", "previous_winner", "winner"]]
                             .assign(predicted_role=predicted_roles, predicted_destination=predicted_parties))

results = pd.DataFrame(metric_rows).set_index("election")
predictions = pd.concat(prediction_frames, ignore_index=True)
validation_folds = pd.DataFrame(fold_rows)


## Held-out destination accuracy

Each row evaluates **only actual changed seats** in the named election. `changed_seat_accuracy` is `n_correct / n_changed_evaluation`: the proportion whose new winning party was predicted correctly. Higher is better. `always_contesting_accuracy` is a baseline that always selects the strongest previous challenger, so the fitted model can be compared against a simple destination rule.

`n_changed_train` counts all earlier changed-seat observations used for the final refit. `best_max_depth` and `best_n_estimators` report the selected settings. `mean_cv_accuracy` is the tuning score averaged across `n_inner_folds` earlier validation elections; it is not the held-out score. Small evaluation samples and rare destination classes can make accuracy variable across elections.


In [5]:
display(results.round(4))
pooled_accuracy = predictions["predicted_destination"].eq(predictions["winner"]).mean()
print(f"Pooled changed-seat accuracy: {pooled_accuracy:.4f} ({int(results.n_correct.sum())}/{len(predictions)} seats)")
print(f"Mean election accuracy: {results.changed_seat_accuracy.mean():.4f}")
assert len(predictions) == results.n_changed_evaluation.sum()
assert np.isclose(pooled_accuracy, results.n_correct.sum() / results.n_changed_evaluation.sum())


Pooled changed-seat accuracy: 0.8321 (347/417 seats)
Mean election accuracy: 0.8573


,n_changed_train,n_changed_evaluation,n_inner_folds,best_max_depth,best_n_estimators,mean_cv_accuracy,n_correct,changed_seat_accuracy,always_contesting_accuracy
election,,,,,,,,,
2005,277,57,3,3,200,0.9358,54,0.9474,0.8772
2010,334,111,4,3,200,0.9387,103,0.9279,0.9640
2015,445,109,5,3,200,0.9365,58,0.5321,0.7798
2017,554,65,6,2,25,0.9040,58,0.8923,0.8923
2019,619,75,7,2,25,0.9024,74,0.9867,0.9867


Pooled accuracy weights each changed-seat observation equally; mean election accuracy weights each evaluation election equally. These conditional scores do not measure seat-change detection or overall winner accuracy across unchanged seats.

`models[year]` contains the fitted pipeline, `searches[year].cv_results_` retains every grid-search score, and `validation_folds` records the chronological splits. Constituency predictions remain available in `predictions` without displaying a detailed preview table.